# Sample: muestreo y trazabilidad

Laboratorio ESPE · Semilla 42 · Fuentes y límites documentados.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'data' / 'dataset.csv').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
df = pd.read_csv(ROOT / 'data/dataset.csv', keep_default_na=False)
df['labels'] = df.labels.map(json.loads)
print('Raíz del laboratorio:', ROOT)


Raíz del laboratorio: D:\ESPE\36902-DESARROLLO SOFTWARE SEGURO\Pacial2\Trabajo


Objetivo: verificar el muestreo controlado, etiquetas, procedencia y separación por familias antes de entrenar. Hipótesis: separar plantillas completas reduce fuga por variantes, aunque persiste similitud semántica.

In [2]:
display(df[['id','project','language','vulnerable','labels','source','split']].head(8))
display(df.groupby(['split','language','vulnerable']).size().rename('registros').to_frame())

,id,project,language,vulnerable,labels,source,split
0,VM-0001,controlled-python-sql-family0,Python,1,[A05:2025],original-controlled-v1,train
1,VM-0002,controlled-python-sql-family0,Python,0,[],original-controlled-v1,train
2,VM-0003,controlled-python-sql-family0,Python,1,[A05:2025],original-controlled-v1,train
3,VM-0004,controlled-python-sql-family0,Python,0,[],original-controlled-v1,train
4,VM-0005,controlled-python-sql-family0,Python,1,[A05:2025],original-controlled-v1,train
5,VM-0006,controlled-python-sql-family0,Python,0,[],original-controlled-v1,train
6,VM-0007,controlled-python-sql-family0,Python,1,[A05:2025],original-controlled-v1,train
7,VM-0008,controlled-python-sql-family0,Python,0,[],original-controlled-v1,train


registros
split      language   vulnerable           
test       JavaScript 0                  42
                      1                  42
           Python     0                  42
                      1                  42
train      JavaScript 0                 168
                      1                 168
           Python     0                 168
                      1                 168
validation JavaScript 0                  42
                      1                  42
           Python     0                  42
                      1                  42

In [3]:
assert len(df) >= 500
assert (df.vulnerable == 0).sum() >= 100
assert not df.sha256.duplicated().any()
assert df.groupby('project').split.nunique().max() == 1
display(pd.read_csv(ROOT / 'data/cwe_owasp.csv'))
print('Registros:', len(df), '| Corregidos:', (df.vulnerable == 0).sum(), '| Grupos artificiales:', df.project.nunique())

,scenario,cwe,owasp,basis
0,sql,CWE-89,A05:2025,Relacion contextual del escenario; ver docs/fu...
1,crypto,CWE-327,A04:2025,Relacion contextual del escenario; ver docs/fu...
2,debug,CWE-489,A02:2025;API8:2023,Relacion contextual del escenario; ver docs/fu...
3,jwt,CWE-347,A07:2025;API2:2023,Relacion contextual del escenario; ver docs/fu...
4,ssrf,CWE-918,A01:2025;API7:2023,Relacion contextual del escenario; ver docs/fu...
5,deserialize,CWE-502,A08:2025,Relacion contextual del escenario; ver docs/fu...
6,tls,CWE-295,A07:2025;API10:2023,Relacion contextual del escenario; ver docs/fu...


Registros: 1008 | Corregidos: 504 | Grupos artificiales: 84


Conclusión: 1.008 registros, 504 corregidos y 84 grupos artificiales. Las seis variantes de una familia permanecen juntas. Particiones 672/168/168: cuatro familias de entrenamiento, una de validación y una de prueba. Se identificaron OWASP Benchmark y NIST SARD, pero no se incorporó ningún registro suyo; ver `docs/fuentes.md`. El dataset no representa la diversidad ni la prevalencia de producción.